In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = "/content/drive/MyDrive/behavior-aware-bms"

battery = pd.read_csv(
    f"{PROJECT_ROOT}/data/features/battery_summary_v1.csv"
)

battery.head()

,battery_id,avg_stress,avg_temp,fast_charge_duration,deep_discharge_duration,high_temp_duration,aggressive_discharge_count,avg_soc
0,B0005,13.670454,26.369701,1.250087e+09,1.277054e+07,3581775.549,45284,NaN
1,B0006,11.407560,26.429154,1.027942e+09,1.931800e+07,4680506.022,44512,NaN
2,B0007,0.065179,26.119363,6.213354e+05,1.428340e+07,4420739.468,195,NaN
3,B0018,14.449429,25.913199,6.962533e+08,1.054231e+07,0.000,32084,NaN
4,B0025,16.337941,28.482356,2.693701e+08,1.155482e+07,2827773.483,4349,NaN


In [5]:
print(battery.columns.tolist())

['battery_id', 'avg_stress', 'avg_temp', 'fast_charge_duration', 'deep_discharge_duration', 'high_temp_duration', 'aggressive_discharge_count', 'avg_soc']


In [6]:
battery["risk_score"] = 0

In [7]:
battery["risk_score"] += np.where(
    battery["avg_stress"] >= 70,
    30,
    np.where(
        battery["avg_stress"] >= 50,
        20,
        10
    )
)

In [9]:
battery["risk_score"] += np.where(
    battery["avg_temp"] >= 40,
    25,
    np.where(
        battery["avg_temp"] >= 30,
        15,
        5
    )
)

In [10]:
battery["risk_score"] += np.where(
    battery["deep_discharge_duration"] >= 100,
    20,
    np.where(
        battery["deep_discharge_duration"] >= 20,
        10,
        5
    )
)

In [11]:
battery["risk_score"] += np.where(
    battery["fast_charge_duration"] >= 100,
    15,
    np.where(
        battery["fast_charge_duration"] >= 20,
        8,
        2
    )
)

In [12]:
battery["risk_score"] += np.where(
    battery["aggressive_discharge_count"] >= 500,
    15,
    np.where(
        battery["aggressive_discharge_count"] >= 100,
        8,
        2
    )
)

In [13]:
battery["risk_score"] += np.where(
    battery["avg_soc"] > 80,
    10,
    np.where(
        battery["avg_soc"] < 20,
        10,
        0
    )
)

In [14]:
battery["risk_score"] = battery["risk_score"].clip(0,100)

In [15]:
def classify(score):

    if score >= 80:
        return "CRITICAL"

    elif score >= 60:
        return "HIGH"

    elif score >= 40:
        return "MEDIUM"

    else:
        return "LOW"


battery["risk_level"] = battery["risk_score"].apply(classify)

In [16]:
def generate_reason(row):

    reasons=[]

    if row.avg_stress > 50:
        reasons.append("high stress")

    if row.avg_temp > 30:
        reasons.append("high temperature")

    if row.deep_discharge_duration > 20:
        reasons.append("deep discharge")

    if row.fast_charge_duration > 20:
        reasons.append("frequent fast charging")

    if row.aggressive_discharge_count > 100:
        reasons.append("aggressive discharge")

    if row.avg_soc > 80:
        reasons.append("high SOC exposure")

    if row.avg_soc < 20:
        reasons.append("low SOC exposure")

    if len(reasons)==0:
        reasons.append("healthy usage")

    return ", ".join(reasons)


battery["risk_reason"] = battery.apply(
    generate_reason,
    axis=1
)

In [17]:
def recommendation(row):

    if row.risk_level=="CRITICAL":
        return (
            "Immediate inspection recommended. "
            "Avoid fast charging and maintain 20-80 SOC."
        )

    elif row.risk_level=="HIGH":
        return (
            "Reduce fast charging frequency and "
            "follow 20-80 charging rule."
        )

    elif row.risk_level=="MEDIUM":
        return (
            "Avoid deep discharge and prolonged "
            "high SOC exposure."
        )

    else:
        return (
            "Battery behavior is healthy."
        )


battery["recommended_action"] = battery.apply(
    recommendation,
    axis=1
)

In [18]:
battery[
    [
        "battery_id",
        "risk_score",
        "risk_level",
        "risk_reason",
        "recommended_action"
    ]
].head(20)

,battery_id,risk_score,risk_level,risk_reason,recommended_action
0,B0005,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
1,B0006,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
2,B0007,63,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
3,B0018,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
4,B0025,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
5,B0026,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
6,B0027,57,MEDIUM,"deep discharge, aggressive discharge",Avoid deep discharge and prolonged high SOC ex...
7,B0028,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
8,B0029,100,CRITICAL,"high temperature, deep discharge, frequent fas...",Immediate inspection recommended. Avoid fast c...
9,B0030,100,CRITICAL,"high temperature, deep discharge, frequent fas...",Immediate inspection recommended. Avoid fast c...


In [19]:
risk_distribution = (
    battery["risk_level"]
    .value_counts()
)

print(risk_distribution)

risk_level
HIGH        17
MEDIUM      10
CRITICAL     7
Name: count, dtype: int64


In [20]:
from pathlib import Path

Path(
    f"{PROJECT_ROOT}/data/features"
).mkdir(
    parents=True,
    exist_ok=True
)

battery.to_csv(
    f"{PROJECT_ROOT}/data/features/battery_risk_assessment_v1.csv",
    index=False
)

print("battery risk assessment saved")

battery risk assessment saved


In [22]:
risk_distribution.to_csv(
    f"{PROJECT_ROOT}/reports/metrics/risk_distribution.csv"
)

print("risk distribution saved")

risk distribution saved


In [23]:
risk_doc = """
# T17 Risk Classification Rules

Risk Categories

LOW:
0-39

MEDIUM:
40-59

HIGH:
60-79

CRITICAL:
80-100

Features Used:
- average stress
- average temperature
- deep discharge duration
- fast charge duration
- aggressive discharge count
- average SOC

Recommendations:
- Follow 20-80 charging rule
- Reduce fast charging
- Avoid deep discharge
- Reduce thermal exposure
"""

with open(
    f"{PROJECT_ROOT}/docs/risk_rules.md",
    "w"
) as f:
    f.write(risk_doc)

print("documentation saved")

documentation saved


In [24]:
battery[
    [
        "battery_id",
        "risk_score",
        "risk_level",
        "risk_reason",
        "recommended_action"
    ]
].head(10)

,battery_id,risk_score,risk_level,risk_reason,recommended_action
0,B0005,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
1,B0006,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
2,B0007,63,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
3,B0018,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
4,B0025,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
5,B0026,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
6,B0027,57,MEDIUM,"deep discharge, aggressive discharge",Avoid deep discharge and prolonged high SOC ex...
7,B0028,70,HIGH,"deep discharge, frequent fast charging, aggres...",Reduce fast charging frequency and follow 20-8...
8,B0029,100,CRITICAL,"high temperature, deep discharge, frequent fas...",Immediate inspection recommended. Avoid fast c...
9,B0030,100,CRITICAL,"high temperature, deep discharge, frequent fas...",Immediate inspection recommended. Avoid fast c...


In [26]:
%cd /content/drive/MyDrive/behavior-aware-bms

!git add notebooks/n05_T1v1.ipynb

!git add data/features/battery_risk_assessment_v1.csv

!git add reports/metrics/risk_distribution.csv

!git add docs/risk_rules.md

/content/drive/MyDrive/behavior-aware-bms


In [27]:
!git status

Refresh index: 100% (73/73), done.
On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/features/battery_risk_assessment_v1.csv
	new file:   docs/risk_rules.md
	new file:   notebooks/n05_T1v1.ipynb
	new file:   reports/metrics/risk_distribution.csv

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/04_feature_refinement.ipynb
	modified:   notebooks/BMS_Project_Setup.ipynb
	modified:   notebooks/error_analysis.ipynb
	modified:   notebooks/n05_T1v1.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/nasa_preproc



In [28]:
%cd /content/drive/MyDrive/behavior-aware-bms

!git add notebooks/n05_T1v1.ipynb

/content/drive/MyDrive/behavior-aware-bms


In [29]:
!git status

On branch main
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   data/features/battery_risk_assessment_v1.csv
	new file:   docs/risk_rules.md
	new file:   notebooks/n05_T1v1.ipynb
	new file:   reports/metrics/risk_distribution.csv

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/04_feature_refinement.ipynb
	modified:   notebooks/BMS_Project_Setup.ipynb
	modified:   notebooks/error_analysis.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/nasa_preproc



In [30]:
!git commit -m "Day 17: add battery risk classification engine V1"

[main 87c3e71] Day 17: add battery risk classification engine V1
 4 files changed, 70 insertions(+)
 create mode 100644 data/features/battery_risk_assessment_v1.csv
 create mode 100644 docs/risk_rules.md
 create mode 100644 notebooks/n05_T1v1.ipynb
 create mode 100644 reports/metrics/risk_distribution.csv


In [31]:
!git push origin main

fatal: could not read Username for 'https://github.com': No such device or address


In [32]:
!git remote -v


origin	https://github.com/Navvu-gityhub/Behavior-Aware-BMS.git (fetch)
origin	https://github.com/Navvu-gityhub/Behavior-Aware-BMS.git (push)


In [33]:
import getpass

token = getpass.getpass("GitHub PAT: ")

!git remote set-url origin https://{token}@github.com/Navvu-gityhub/Behavior-Aware-BMS.git

!git push origin main

GitHub PAT: ··········
Enumerating objects: 23, done.
Counting objects: 100% (23/23), done.
Delta compression using up to 2 threads
Compressing objects: 100% (14/14), done.
Writing objects: 100% (15/15), 10.23 KiB | 387.00 KiB/s, done.
Total 15 (delta 5), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (5/5), completed with 4 local objects.
To https://github.com/Navvu-gityhub/Behavior-Aware-BMS.git
   1d07239..87c3e71  main -> main


In [34]:
!git remote set-url origin https://github.com/Navvu-gityhub/Behavior-Aware-BMS.git

In [35]:
!git log --oneline -3
!git status

87c3e71 (HEAD -> main, origin/main) Day 17: add battery risk classification engine V1
9b2e1f4 chore: ignore generated datasets
1d07239 Day 16: add advanced battery behavior features
On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/04_feature_refinement.ipynb
	modified:   notebooks/BMS_Project_Setup.ipynb
	modified:   notebooks/error_analysis.ipynb
	modified:   notebooks/n05_T1v1.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/nasa_preproc

no changes added to commit (use "git add" and/or "git commit -a")


In [36]:
!git commit -m "Day 17: add battery risk classification engine"

On branch main
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/04_feature_refinement.ipynb
	modified:   notebooks/BMS_Project_Setup.ipynb
	modified:   notebooks/error_analysis.ipynb
	modified:   notebooks/n05_T1v1.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	notebooks/nasa_preproc

no changes added to commit (use "git add" and/or "git commit -a")
